In [2]:
import cygnet as cy
import numpy as np
import uproot, gc, os
import imageio.v3 as iio
import matplotlib.pyplot as plt

def debug_plot(noisy, mask):
    plt.figure(figsize = (12,12))
    plt.title(f"MASKED")
    plt.imshow(noisy, vmin = 198, vmax = 220)
    plt.imshow(mask, alpha=0.5)
    plt.show()

# folders:
# input
# mask
# mask_exp2
# mask_exp4

train_path = "/home/frx/dataset/image_dataset/train"
vali_path = "/home/frx/dataset/image_dataset/vali"
test_path = "/home/frx/dataset/image_dataset/test"

data_path = "/home/frx/dataset/root_dataset"

folders = os.listdir(data_path)
files_dict = {}

for fol in folders:
    sub_path = os.path.join(data_path, fol)
    files_dict[fol] = sorted(os.listdir(sub_path))

display(files_dict)

ufile_sample = uproot.open(data_path + "/" + fol + "/" + files_dict[fol][0])
tree = ufile_sample["event_info"]

n_classi = len(folders)
n_file = len(files_dict[fol])
n_eventi = len(tree["eventnumber"].arrays(library="np")["eventnumber"])

print(f"i see {n_classi} class folders each containing {n_file} files, each containing {n_eventi} events.")

i = 0
j = 0
k = 0

train = 0
vali = 0
test = 0

tot = n_classi * n_file * n_eventi
print(f"i see a total of {tot} events")
printable = []

ffol = folders[0]
fffile = files_dict[ffol][0]

factor = 4
exp_radius = 4

in_t = cy.input_transform_builder(factor)
out_t = cy.target_transform_builder(factor)
exp2_t = cy.inflate_transform_builder(2)
exp4_t = cy.inflate_transform_builder(4)

for fol, files in files_dict.items():

    for j, ffile in enumerate(files):
        if j < 3:
            train_set = True
        else:
            train_set = False

        fpath = os.path.join(data_path, fol, ffile)
        with uproot.open(fpath) as ufile:
            tree = ufile["event_info"]
            redxs = tree["redpix_ix"].arrays(library="np")["redpix_ix"]
            redys = tree["redpix_iy"].arrays(library="np")["redpix_iy"]
            redzs = 255 #tree["redpix_iz"].arrays(library="np")["redpix_iz"]
            p_shape = ufile[f"pic_run{j+1}_ev{0}"].to_numpy()[0].shape

            for k in range(n_eventi):
                print(fol, ffile, train, vali, test)
                noisy = ufile[f"pic_run{j+1}_ev{k}"].to_numpy()[0].T
                xe = redxs[k]
                ye = redys[k]
                mask = np.zeros(p_shape, dtype=np.uint8)
                mask[xe, ye] = redzs
                mask = mask.T

                noisy = in_t(noisy).squeeze()
                mask = out_t(mask).squeeze()
                exp2_mask = exp2_t(mask).squeeze()
                exp4_mask = exp4_t(mask).squeeze()

                if train_set:
                    iio.imwrite(os.path.join(train_path, "input", f"i_{train}.png"), noisy, extension=".png")
                    iio.imwrite(os.path.join(train_path, "mask", f"m_{train}.png"), mask, extension=".png")
                    iio.imwrite(os.path.join(train_path, "mask_exp2", f"m2_{train}.png"), exp2_mask, extension=".png")
                    iio.imwrite(os.path.join(train_path, "mask_exp4", f"m4_{train}.png"), exp4_mask, extension=".png")

                    train += 1

                else:
                    if k < 100:
                        iio.imwrite(os.path.join(vali_path, "input", f"i_{vali}.png"), noisy, extension=".png")
                        iio.imwrite(os.path.join(vali_path, "mask", f"m_{vali}.png"), mask, extension=".png")
                        iio.imwrite(os.path.join(vali_path, "mask_exp2", f"m2_{vali}.png"), exp2_mask, extension=".png")
                        iio.imwrite(os.path.join(vali_path, "mask_exp4", f"m4_{vali}.png"), exp4_mask, extension=".png")
                        vali += 1
                    else:
                        iio.imwrite(os.path.join(test_path, "input", f"i_{test}.png"), noisy, extension=".png")
                        iio.imwrite(os.path.join(test_path, "mask", f"m_{test}.png"), mask, extension=".png")
                        iio.imwrite(os.path.join(test_path, "mask_exp2", f"m2_{test}.png"), exp2_mask, extension=".png")
                        iio.imwrite(os.path.join(test_path, "mask_exp4", f"m4_{test}.png"), exp4_mask, extension=".png")
                        test += 1

                noisy = None
                mask = None
                exp2_mask = None
                exp4_mask = None
                gc.collect()

            tree = None
            redxs = None
            redys = None
            gc.collect()

print()
print(f"N eventi: {tot}")
print(f"dataset split: {train} / {vali} / {test}")
print(f"o, in proporzione: {train/tot} / {vali/tot} / {test/tot}")

{'Acrylic_K40_TracceLunghe': ['histograms_Run00001_Quest2.root',
  'histograms_Run00002_Quest2.root',
  'histograms_Run00003_Quest2.root',
  'histograms_Run00004_Quest2.root'],
 'ER_50_keV': ['histograms_Run00001_Quest2.root',
  'histograms_Run00002_Quest2.root',
  'histograms_Run00003_Quest2.root',
  'histograms_Run00004_Quest2.root'],
 'He_5_keVee': ['histograms_Run00001_Quest2.root',
  'histograms_Run00002_Quest2.root',
  'histograms_Run00003_Quest2.root',
  'histograms_Run00004_Quest2.root'],
 'He_50_keVee': ['histograms_Run00001_Quest2.root',
  'histograms_Run00002_Quest2.root',
  'histograms_Run00003_Quest2.root',
  'histograms_Run00004_Quest2.root'],
 'He_15_keVee': ['histograms_Run00001_Quest2.root',
  'histograms_Run00002_Quest2.root',
  'histograms_Run00003_Quest2.root',
  'histograms_Run00004_Quest2.root'],
 'ER_15_keV': ['histograms_Run00001_Quest2.root',
  'histograms_Run00002_Quest2.root',
  'histograms_Run00003_Quest2.root',
  'histograms_Run00004_Quest2.root'],
 'ER_5_k

i see 7 class folders each containing 4 files, each containing 200 events.
i see a total of 5600 events
Acrylic_K40_TracceLunghe histograms_Run00001_Quest2.root 0 0 0
Acrylic_K40_TracceLunghe histograms_Run00001_Quest2.root 1 0 0
Acrylic_K40_TracceLunghe histograms_Run00001_Quest2.root 2 0 0
Acrylic_K40_TracceLunghe histograms_Run00001_Quest2.root 3 0 0
Acrylic_K40_TracceLunghe histograms_Run00001_Quest2.root 4 0 0
Acrylic_K40_TracceLunghe histograms_Run00001_Quest2.root 5 0 0
Acrylic_K40_TracceLunghe histograms_Run00001_Quest2.root 6 0 0
Acrylic_K40_TracceLunghe histograms_Run00001_Quest2.root 7 0 0
Acrylic_K40_TracceLunghe histograms_Run00001_Quest2.root 8 0 0
Acrylic_K40_TracceLunghe histograms_Run00001_Quest2.root 9 0 0
Acrylic_K40_TracceLunghe histograms_Run00001_Quest2.root 10 0 0
Acrylic_K40_TracceLunghe histograms_Run00001_Quest2.root 11 0 0
Acrylic_K40_TracceLunghe histograms_Run00001_Quest2.root 12 0 0
Acrylic_K40_TracceLunghe histograms_Run00001_Quest2.root 13 0 0
Acrylic_K4

In [3]:
train_inputs = os.listdir(os.path.join(train_path, "input"))
train_masks = os.listdir(os.path.join(train_path, "mask"))
train_2emasks = os.listdir(os.path.join(train_path, "mask_exp2"))
train_4emasks = os.listdir(os.path.join(train_path, "mask_exp4"))

vali_inputs = os.listdir(os.path.join(vali_path, "input"))
vali_masks = os.listdir(os.path.join(vali_path, "mask"))
vali_2emasks = os.listdir(os.path.join(vali_path, "mask_exp2"))
vali_4emasks = os.listdir(os.path.join(vali_path, "mask_exp4"))

test_inputs = os.listdir(os.path.join(test_path, "input"))
test_masks = os.listdir(os.path.join(test_path, "mask"))
test_2emasks = os.listdir(os.path.join(test_path, "mask_exp2"))
test_4emasks = os.listdir(os.path.join(test_path, "mask_exp4"))

print(len(train_inputs))
print(len(train_masks))
print(len(train_2emasks))
print(len(train_4emasks))

print(len(vali_inputs))
print(len(vali_masks))
print(len(vali_2emasks))
print(len(vali_4emasks))

print(len(test_inputs))
print(len(test_masks))
print(len(test_2emasks))
print(len(test_4emasks))

4200
4200
4200
4200
700
700
700
700
700
700
700
700
